In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm

import sys

sys.path.append('../src')

from Utils.utils import PPT
from Config.config import PATHS
from Utils.cherrypick_simulations import CherryPickEquilibria
from Classes.focal_regions import SetFocalRegions, FocalRegion

In [2]:
file_name = PATHS['human_data'] / 'multi-player.csv'
data = pd.read_csv(file_name)

In [3]:
list_fixed_parameters = PPT.get_fixed_parameters(data)
# list_fixed_parameters = [fp for fp in list_fixed_parameters if fp['num_agents'] in [2, 3, 4]]
num_groups = len(list_fixed_parameters)
print(f'{len(list_fixed_parameters)} fixed parameters and {len(list_fixed_parameters)} simulation parameters')

27 fixed parameters and 27 simulation parameters


In [ ]:
max_regions = 12

for fixed_parameters in tqdm(list_fixed_parameters, desc='Fixed parameters'):
    print("Fixed parameters:\n", fixed_parameters)
    cp = CherryPickEquilibria(
        num_agents=fixed_parameters['num_agents'],
        threshold=fixed_parameters['threshold'],
        epsilon=0,
        num_rounds=fixed_parameters['num_agents'],
        allow_shuffle=True,
    )
    sfr = SetFocalRegions(
        num_agents=fixed_parameters['num_agents'],
        threshold=fixed_parameters['threshold'],
        len_history=fixed_parameters['num_agents'],
        max_regions=max_regions,
    )
    list_regions = []
    counter = 0
    while counter < max_regions:
        print(f'Generating fair focal region...')
        region = cp.random_fair_periodic_equilibrium(period=fixed_parameters['num_agents'])
        # FocalRegion.draw_region(region)
        region = FocalRegion(region, 'fair')
        list_regions.append(region)
        counter += 1
        if counter == max_regions:
            break
        print(f'Generating segmented focal region...')
        region = cp.random_periodic_equilibrium(period=fixed_parameters['num_agents'])
        # FocalRegion.draw_region(region)
        region = FocalRegion(region, 'segmented')
        list_regions.append(region)
        counter += 1
        if counter == max_regions:
            break
        if sfr.B > 1:
            print(f'Generating mixed focal region...')
            region = cp.random_mixed_periodic_equilibrium(period=fixed_parameters['num_agents'])
            # FocalRegion.draw_region(region)
            region = FocalRegion(region, 'mixed')
            list_regions.append(region)
            counter += 1
            if counter == max_regions:
                break
    sfr.focal_regions = list_regions
    # for region in sfr.focal_regions:
    #     FocalRegion.draw_region(region)
    sfr.file = PATHS['focal_regions_path'] / f"_{sfr.num_agents}_agents_{sfr.threshold}_threshold.json"
    sfr.save_focal_regions()

Fixed parameters:   0%|          | 0/27 [00:00<?, ?it/s]

Fixed parameters:
 {'num_agents': 4, 'threshold': 0.5}
Generating fair focal region...
Generating segmented focal region...
Generating mixed focal region...
Fixed parameters:
 {'num_agents': 4, 'threshold': 0.75}
Generating fair focal region...
Generating segmented focal region...
Generating mixed focal region...
Fixed parameters:
 {'num_agents': 4, 'threshold': 0.25}
Generating fair focal region...
Generating segmented focal region...
Generating fair focal region...
Fixed parameters:
 {'num_agents': 8, 'threshold': 0.375}
Generating fair focal region...
Generating segmented focal region...
Generating mixed focal region...
Fixed parameters:
 {'num_agents': 8, 'threshold': 0.625}
Generating fair focal region...
Generating segmented focal region...
Generating mixed focal region...
Fixed parameters:
 {'num_agents': 8, 'threshold': 0.875}
Generating fair focal region...
Generating segmented focal region...
Generating mixed focal region...
Fixed parameters:
 {'num_agents': 9, 'threshold': 0

In [ ]:
import json
import re
from collections import Counter, defaultdict


def region_key(region) -> tuple:
    return tuple(map(tuple, np.asarray(region, dtype=float).round(8)))


pattern = re.compile(r'^_(\d+)_agents_(.+)_threshold\.json$')
rows = []

for path in sorted(PATHS['focal_regions_path'].glob('*.json')):
    match = pattern.match(path.name)
    if match is None:
        continue
    regions = json.loads(path.read_text())
    categories = Counter(r.get('category', 'unknown') for r in regions)
    unique_by_cat = defaultdict(set)
    for r in regions:
        unique_by_cat[r.get('category', 'unknown')].add(region_key(r['region']))
    unique_keys = set().union(*unique_by_cat.values()) if unique_by_cat else set()
    rows.append({
        'num_agents': int(match.group(1)),
        'threshold': float(match.group(2)),
        'n_regions': len(regions),
        'n_unique': len(unique_keys),
        'n_unique_fair': len(unique_by_cat['fair']),
        'n_unique_segmented': len(unique_by_cat['segmented']),
        'n_unique_mixed': len(unique_by_cat['mixed']),
        'n_fair': categories.get('fair', 0),
        'n_segmented': categories.get('segmented', 0),
        'n_mixed': categories.get('mixed', 0),
    })

summary = (
    pd.DataFrame(rows)
    .sort_values(['num_agents', 'threshold'])
    .reset_index(drop=True)
)
summary

,num_agents,threshold,n_regions,n_unique,n_unique_fair,n_unique_segmented,n_unique_mixed,n_fair,n_segmented,n_mixed
0,2,0.500000,12,3,1,2,0,6,6,0
1,3,0.333333,12,4,1,3,0,6,6,0
2,3,0.666667,12,6,1,2,3,4,4,4
3,4,0.250000,12,11,5,6,0,6,6,0
4,4,0.500000,12,12,4,4,4,4,4,4
5,4,0.750000,12,12,4,4,4,4,4,4
6,5,0.400000,12,8,1,3,4,4,4,4
7,5,0.600000,12,12,4,4,4,4,4,4
8,5,0.800000,12,6,1,2,3,4,4,4
9,6,0.333333,12,8,1,3,4,4,4,4
